# Matrix Decompositions — Hands-on

## 1. Eigendecomposition

In [ ]:
import numpy as np

A = np.array([[4, 2], [1, 3]])
eigvals, eigvecs = np.linalg.eig(A)

print("Eigenvalues:", eigvals)
print("Eigenvectors:\n", eigvecs)

## 2. SVD on the iris dataset

In [ ]:
from sklearn.datasets import load_iris
import numpy as np

data = load_iris()
X = data.data
X_centered = X - X.mean(axis=0)

U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
print("Singular values:", S)
print("Explained variance ratio:", (S**2) / np.sum(S**2))

## 3. From-scratch PCA vs sklearn PCA

In [ ]:
from sklearn.decomposition import PCA

# From-scratch: project onto top 2 singular vectors
X_reduced_manual = U[:, :2] * S[:2]

# sklearn
pca = PCA(n_components=2)
X_reduced_sklearn = pca.fit_transform(X)

print("Manual PCA (first 5 rows):\n", X_reduced_manual[:5])
print("\nsklearn PCA (first 5 rows):\n", X_reduced_sklearn[:5])
print("\n(Signs/direction may differ, but magnitudes should match — that's expected with SVD.)")

## 4. Verifying eigendecomposition: A·v = λ·v

In [ ]:
# Confirm the defining property of eigenvectors/eigenvalues
for i in range(len(eigvals)):
    v = eigvecs[:, i]
    lhs = A @ v
    rhs = eigvals[i] * v
    print(f"Eigenvalue {eigvals[i]:.3f}")
    print("A @ v      =", lhs)
    print("lambda * v =", rhs)
    print("Match:", np.allclose(lhs, rhs))
    print()

## 5. Reconstructing A from its eigendecomposition

In [ ]:
# A = V * diag(eigvals) * V^-1
V = eigvecs
Lambda = np.diag(eigvals)
V_inv = np.linalg.inv(V)

A_reconstructed = V @ Lambda @ V_inv
print("Original A:\n", A)
print("\nReconstructed A:\n", np.real(A_reconstructed))
print("\nMatch:", np.allclose(A, A_reconstructed))

## 6. How many components do you actually need? (explained variance)

In [ ]:
import matplotlib.pyplot as plt

explained_var = (S**2) / np.sum(S**2)
cumulative_var = np.cumsum(explained_var)

fig, ax = plt.subplots()
ax.bar(range(1, len(S)+1), explained_var, label='individual')
ax.plot(range(1, len(S)+1), cumulative_var, 'ro-', label='cumulative')
ax.set_xlabel('Component')
ax.set_ylabel('Explained variance ratio')
ax.set_title('How much variance each principal component captures')
ax.legend()
plt.show()

print("With 2 components, we capture", round(cumulative_var[1]*100, 1), "% of the variance")

## 7. Low-rank approximation (compressing a matrix with SVD)

In [ ]:
# Reconstruct X using only the top-k singular values/vectors
def low_rank_approx(U, S, Vt, k):
    return U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

X_rank1 = low_rank_approx(U, S, Vt, k=1)
X_rank2 = low_rank_approx(U, S, Vt, k=2)
X_full  = low_rank_approx(U, S, Vt, k=len(S))

error_rank1 = np.linalg.norm(X_centered - X_rank1)
error_rank2 = np.linalg.norm(X_centered - X_rank2)

print("Reconstruction error using rank-1 approx:", error_rank1)
print("Reconstruction error using rank-2 approx:", error_rank2)
print("\nThis is the same idea used to compress images: keep only the")
print("top-k singular values/vectors and throw away the rest.")